# Import Cell

In [1]:
import torch
import numpy as np
import time
device='cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

# Difference Between GPU and CPU usage

The format specifier :.4f determines how a decimal number (floating-point number) is formatted and displayed in programming languages like Python.
It means the number will be displayed as a fixed-point decimal rounded to exactly 4 decimal places.
Here is how the syntax breaks down:
- : Signals the start of the formatting rules.
- .4 Specifies the precision, forcing exactly 4 digits after the decimal point.
- f Represents "fixed-point" notation, which displays the number as a standard decimal rather than in scientific notation.

We can also use %%time as a header to record the time taken for execution.
What happens realtime is this, cpu executes but there are some waiting times in between blocks of execution. 
```CPU times``` shows cpu calculations only while ```Wall times``` include those wait times as well.

In [2]:
%%time
# we are using time module of python
start_time=time.time() # It will store the time from epoch in ms on execution at this line

# Matrix Operation Here
zeros=torch.zeros(1,1)
end_time=time.time()

delta_time=end_time-start_time
print(f'{delta_time:.4f}')


0.0001
CPU times: user 187 μs, sys: 43 μs, total: 230 μs
Wall time: 229 μs


```0.0000```
- ```CPU times```: user 136 μs, sys: 5 μs, total: 141 μs; Calculates the cpu usage
- ```Wall time```: 134 μs; Calculates the total time from execution and side-effect appearing

## Let's Test which one is faster on gpu, torch or numpy

In [3]:
torch_rand1=torch.rand((10000,10000),dtype=torch.float64).to(device)
# I don't know what to device does. probably assigning it on our global variable 'device' in the import cell
torch_rand2=torch.rand((10000,10000),dtype=torch.float64).to(device)
# Out guy made a mistake, he shoved tensors into np.multiply, apples-to-oranges maths
np_rand1=np.random.rand(10000,10000).astype(np.float64)
# This is np.random composite module method .rand(start=,end=). Here we use astype() incase of detype of torch and use np.float32 object as type
np_rand2=np.random.rand(10000,10000).astype(np.float64)

# Time for cuda block;
start_time=time.time()
rand_gpu=(torch_rand1@torch_rand2)
# The above is the huge multiplication(we don't need to see it), @ is torch specific mul
end_time=time.time()
delta_time=end_time-start_time
print(f'{delta_time:.8f}')
# 0.00019813 fucking screaming fast!
# For float64 0.00018287 SAME SPEED!

# Time for numpy block;
start_time=time.time()
#rand=np.multiply(np_rand1,np_rand2)
# The above is element wise multiplication, we need to compare numpy object matrix multiplication using @
rand_cpu=np_rand1@np_rand2
end_time=time.time()
delta_time=end_time-start_time
print(f'{delta_time:.8f}')
# 4.21621251 screaming slow!
# For float64 10.94090414; double as expected

# Let's time numpy elementwise mul
start_time=time.time()
rand_cpu=np.multiply(np_rand1,np_rand2)
end_time=time.time()
delta_time=end_time-start_time
print(f'{delta_time:.8f}')
# 0.09828210 okay, but definitely not doing the job!
# For float64 0.17952561; double as expected
# Shape is simple, so cpu is better at it!

0.01819062
11.56569886
0.17277265


## Let's Test a shape of (100,100,100,100) so 4D, let's see who wins

In [4]:
torch_rand1=torch.rand((100,100,100,100),dtype=torch.float64).to(device)
# I don't know what to device does. probably assigning it on our global variable 'device' in the import cell
torch_rand2=torch.rand((100,100,100,100),dtype=torch.float64).to(device)
# Out guy made a mistake, he shoved tensors into np.multiply, apples-to-oranges maths
np_rand1=np.random.rand(100,100,100,100).astype(np.float64)
# This is np.random composite module method .rand(start=,end=). Here we use astype() incase of detype of torch and use np.float32 object as type
np_rand2=np.random.rand(100,100,100,100).astype(np.float64)

# Time for cuda block;
start_time=time.time()
rand_gpu=(torch_rand1@torch_rand2)
# The above is the huge multiplication(we don't need to see it), @ is torch specific mul
end_time=time.time()
delta_time=end_time-start_time
print(f'{delta_time:.8f}')
# 0.00027418 slower than 2D!
# For float64 0.00026464 even Faster!!!!

# Time for numpy block;
start_time=time.time()
#rand=np.multiply(np_rand1,np_rand2)
# The above is element wise multiplication, we need to compare numpy object matrix multiplication using @
rand_cpu=np_rand1@np_rand2
end_time=time.time()
delta_time=end_time-start_time
print(f'{delta_time:.8f}')
# 0.16696119 great performance!
# For float64 0.35045409; double as expected

# Let's time numpy elementwise mul
start_time=time.time()
rand_cpu=np.multiply(np_rand1,np_rand2)
end_time=time.time()
delta_time=end_time-start_time
print(f'{delta_time:.8f}')
# 0.09236670 okay, but definitely not doing the job!
# For float64 0.17640281; double as expected
# Shape is simple, so cpu is better at it!

0.00041389
0.39669228
0.19306684


# Billions of But Simple Computation
When we will have millions or billions of parameters for our model, we will be doing very simple computations.
Above example demonstrates real time how efficient my rtx 3060 ti 8GB gpu is compared to 10700k.

## Let's Now Work on Some torch functions

In [5]:
# Define a probability tensor
probabilities=torch.tensor([0.1,0.9])
# 10% or .1->0.90% or .9->1; probabilities[0],probabilities[1]=0.1,0.9 
# Each probability points to the index of the of the probability in the tensor and they have to add up to 1
# Draw 5 samples from multinomial distribution
samples=torch.multinomial(probabilities,num_samples=10,replacement=True)
# Multinomial takes in a probability tensor object of 1 by 2 as its first argument and creates a 1D tensor consisting of 0s and 1s.
# Multinomials shows the indexes in probability order or bias, remember that wighted walk by mewstro?
# According to our script the index 0 has 10% chance to exist and index 1 has 90% chance to exist, and torch.multinomial returns a random 1D tensor with the indexes as elements(booleans for something else?).
print(samples)

tensor([1, 1, 1, 1, 1, 1, 1, 1, 0, 1])


We are going to use this to predict what words gonna come next as biases(or random walks).

In [6]:
# torch.cat() concatenates two tensors like lists
tensor=torch.tensor([1,2,3,4])
out=torch.cat((tensor,torch.tensor([5])),dim=0)
out

tensor([1, 2, 3, 4, 5])

Above torch.cat() is same as the concatenation below.

In [7]:
a=[1,2,3,4]
b=[5]
a+b

[1, 2, 3, 4, 5]

## We Are Going To Actually Use This During Generation
1. It's going to start from zero, we are going to use our probability of distribution to pick the first character.
2. Based on the first character we are going to pick the second one and then ```concatenate``` with the first one.

For example if we have 100 characters here and remaining 100 there, we are going to concatenate them using this method. And in the end we will have all of these integers that we have predicted.

## tril Stands For Triangle Lower(Including The Diagonal)

In [8]:
# tril is gonna multiply the upper part of the diagonal with zero
out=torch.tril(torch.ones(5,5))
out=torch.tril(torch.ones(5,5))
out

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

# The Triangle Shows Time
First we predict 1 and remaining 4 is zero in the first row but slowly the ones talk with the next one and we have 3 zeros and in the last tow we just finished predicting all 5 of them.
```
[[1., 0., 0., 0., 0.],
 [1., 1., 0., 0., 0.],
 [1., 1., 1., 0., 0.],
 [1., 1., 1., 1., 0.],
 [1., 1., 1., 1., 1.]]
```
In the end we have more and more history to look back into rather than future.
## Premise:
Basically the premise of this is to make sure that we can't communicate with the answere, just like in the exam hall. Opposit case for triu;
```
[[1., 1., 1., 1., 1.],
 [0., 1., 1., 1., 1.],
 [0., 0., 1., 1., 1.],
 [0., 0., 0., 1., 1.],
 [0., 0., 0., 0., 1.]]
```
## triu Stands For Triangle Upper(Including The Diagonal)

In [9]:
# triu is gonna multiply the lower part of the diagonal with zero
out=torch.triu(torch.ones(5,5))
out

tensor([[1., 1., 1., 1., 1.],
        [0., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 1.]])

# Mask
When we store a boolean condition with a given tensor containing 1s or 0s, we get a mask, a tensor with trues where boolean satisfies and falses where it doesn't. a mask as such, a tensor of boolean value is insertable into the ```masked_fill()``` method of another tensor object;
```
tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])
```
The Trues get multiplied element-wise with the 2nd args provided as an float() or int() object and gives a tensor as;
```
tensor([[False,  arg2,   arg2,  arg2,  arg2 ],
        [False,  False,  arg2,  arg2,  arg2 ],
        [False,  False,  False, arg2,  arg2 ],
        [False,  False,  False, False, arg2 ],
        [False,  False,  False, False, False]])
```

In [10]:
mask=torch.tril(torch.ones(5,5))==0
masked=torch.zeros(5,5).masked_fill(mask,float('-inf'))
masked

tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])

# Exponentiate Refers to Natural Exoinential Function
Since we were just working with `masked_fill` and `float('-inf')`, exponentiation is the very next step in the **Softmax** formula where $e^{0}=0$;

* **Positive numbers grow fast:** Larger scores become much larger weights.
* **Negative numbers become tiny positive fractions:** For example, $e^{-2} \approx 0.1353$.
* **Negative infinity (`-inf`) becomes `0`:** Because $e^{-\infty} = 0$, filling masked spots with `-inf` ensures they turn into `0.0` after exponentiation and get completely ignored by the model.

In [11]:
# Let's use torch.exp().
torch.exp(masked)

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

## '-inf' Becomes 0 and 0 becomes 1
```
tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])
```
## Transpose is More Like Swapping

In [12]:
input=torch.zeros(2,3,4)
# By default torch.zeros() takes dimension as args.
# We can use method transpose() to swap the dimension value from the input tensor.
# .transpose() takes is index th number of the dimensions and swaps them.
out=input.transpose(0,2)
# Here we just swaped 0th dimension 2 with 2th dimention 4.
input,out

(tensor([[[0., 0., 0., 0.],
          [0., 0., 0., 0.],
          [0., 0., 0., 0.]],
 
         [[0., 0., 0., 0.],
          [0., 0., 0., 0.],
          [0., 0., 0., 0.]]]),
 tensor([[[0., 0.],
          [0., 0.],
          [0., 0.]],
 
         [[0., 0.],
          [0., 0.],
          [0., 0.]],
 
         [[0., 0.],
          [0., 0.],
          [0., 0.]],
 
         [[0., 0.],
          [0., 0.],
          [0., 0.]]]))

# Stack Tensors on Next Dimension
torch.stack([torch.tensor]) takes in tensor objects and house them serially as rows in the direction of the column space. It can take in tensors as opposed to torch.tensor([list(]) which takes in list objects only.
- Same as multiplying a vector $(\vec{a}_j-y_i)$ with a transpose $(\vec{a}_j^0)^T$ to get a 2d matrix.

We can stack from $1D\rightarrow 2D$ or any $(n-1)D\rightarrow nD$

## Relevance to Bach Size

In [13]:
tensor1=torch.tensor([1,2,3])
tensor2=torch.tensor([4,5,6])
tensor3=torch.tensor([7,8,9])

# Stack along next dimension
stacked=torch.stack([tensor1,tensor2,tensor3])
stacked

tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]])

# Neural Network NN
It's very important for the model it's called the nn.linear function. nn.Linear() has weights and biases and it learns and gets better.
Let's go to torch.org torch.nn documentation. We have containers and a lot of layers.
For our example with nn.Linear(), we will start with a 3 $\vec{a}_3^0$ inputs and 3 $\vec{a}_3^L$ outputs; usually vectors of equal length $(\vec{a}^0_j\rightarrow\vec{a}_j^L)$ and compute;
$$W^L_{(3,3)}\times \vec{a}_3^0+b^L=\vec{a}_3^L$$
For nn.Sequential() has in between layers, and we will have to sync our $\vec{W}^l_{ij}$ weights with each $\vec{a}_3^{l-1},\vec{a}_3^l$ so that math can do the mathing.

In [14]:
import torch.nn as nn
sample=torch.tensor([10.,10.,10.])
# Must be floats
linear=nn.Linear(3,3,bias=False)
linear(sample)

tensor([-0.5711,  4.7244,  3.3144], grad_fn=<SqueezeBackward4>)

# Softmax Function
Softmax function isn't actually what it sounds like at all. When we shove our tensor into it we are doing exactly this to each of the tensor members; 
$$tensor[i]'=\frac{e^{tensor[i]}}{\sum_{i=1}^m e^{tensor[i]}}$$
In more accurate language the standard unit softmax function $\sigma:\mathbb{R}^K\rightarrow(0,1)^K$ is defined when $K\ge1$ by the formula
$$\sigma(z)_i=\frac{e^{z_i}}{\sum_{i=1}^m e^{z_i}}\text{ for }i=1,...,K\text{ and z}=(z_1,...,z_K)\in\mathbb{R}^K$$
This is also called normalization or an activation function as opposed to relu or max(0,1).

In [15]:
import torch.nn.functional as F
# Create a tensor
tensor1=torch.tensor([1.,2.,3.])
# Apply softmax using torch.nn.functional.softmax()
softmax_output=F.softmax(tensor1,dim=0) # Scalar?
softmax_output

tensor([0.0900, 0.2447, 0.6652])

# nn.Embedding 
It is a class in torch that represents an embedding layer which represents a vecor of discrete inputs, such as words as categorical variables, as a continuous stream of long vector of values, our $\vec{a}^0_j$ layer.
The nn.Embedding layer maps each discrete input to a dense vector representation, often referred to as an embedding vector. They capture the semantic relationships between the inputs, allowing the model to learn meaningful representation for the discrete inputs. Basically numeric representation in a simple long vector.
## The gpt Generated Code as an Example Below

In [16]:
# initialize an embedding layer
vocab_size=1000
embedding_dim=100 # Number of elements per stack
embedding=nn.Embedding(vocab_size,embedding_dim) # The so called argument it takes in
# Create some input indices, just random
input_indices=torch.LongTensor([1,5,3,2]) # Note that it is made of int64 as indices and determines the number of stacks.
# Apply the embedding layer
embedded_output=embedding(input_indices) # It only takes in ints so int64
# These nn.blablas are functions assigned like objects where variables latter produce () s
embedded_output.shape
# cell output <torch.Size([4, 100])>, sampling fot large values?

torch.Size([4, 100])

# gpt Explains
For example let's say we have a small vocabulary with just four words; ['cat','dog','apple','banana'] and we want to convert these words into embedding vector of size 2 i.e, $2D$ vecors;
| Word | Embedding Vector |
| :--- | :--- |
| 'cat' | [0.2,-0.8] |
| 'dog' | [0.4,-0.1] |
| 'apple' | [0.6,-0.3] |
| 'banana' | [0.7,-0.9] |

The above was word level, so now let's consider character level embedding where we take 26 lowercase letters and embed vectors of size 5 for each character;
| Character | Embedding Vector |
| :--- | :--- |
| 'a' | [0.3319, 0.9057, 0.4302, 0.0179, 0.4178] |
| 'b' | [0.5664, 0.3944, 0.6879, 0.1338, 0.1401] |
| 'c' | [0.4779, 0.6485, 0.3951, 0.4873, 0.1529] |
$\cdots$
| 'z' | [0.2737, 0.1350, 0.6850, 0.1797, 0.9680] |

The embedding vector is the numerical representation of the sentiment of a letter or any quality that we assign or let our model to assign.

In [33]:
import random
# torch.rand() doesnt allow int so use randint
a=torch.randint(10,(3,2))
# arg1 is the range of selection like random.randint() and the tuple is the shape
b=torch.randint(10,(2,3))
[a,b,(a@b),torch.matmul(a,b)]
# @ denotes the matmul, no pun intended, remember jacobian chain rule and compute per node?

[tensor([[8, 9],
         [7, 1],
         [1, 8]]),
 tensor([[0, 9, 1],
         [4, 1, 8]]),
 tensor([[36, 81, 80],
         [ 4, 64, 15],
         [32, 17, 65]]),
 tensor([[36, 81, 80],
         [ 4, 64, 15],
         [32, 17, 65]])]

In [44]:
int_64=torch.randint(4,(3,2))
# type int64
float_32=torch.rand(2,3)
# type float32
#result=torch.matmul(int_64,float_32)
# We can not multiply different datatypes.
# It will get this error, <RuntimeError: expected m1 and m2 to have the same dtype, but got: long int != float>
# We can solve this by casting float method on the int_64 
result=torch.matmul(int_64.float(),float_32)
result
# Casting .float() is not the best, it strips away a lot of prescision.

tensor([[0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000],
        [1.9362, 2.1962, 0.6580]])

# Multilateral Perceptron
Yeah, that's what NNs are called. Now let's go back to our bigram.ipynb notebook

In [18]:
torch.rand(5,1)

tensor([[0.3582],
        [0.6898],
        [0.7687],
        [0.5217],
        [0.0010]])